In [8]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager


# ==========================================================
# 1. Configuración del navegador
# ==========================================================
def iniciar_navegador():
    options = Options()

    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    options.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )

    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

    return driver


# ==========================================================
# 2. Datos a modificar según el sitio web
# ==========================================================
URL_OBJETIVO = "https://books.toscrape.com/"

SELECTOR_CONTENEDOR = "article.product_pod"
SELECTOR_DATO_A = "p.price_color"
SELECTOR_DATO_B = "p.star-rating"


# ==========================================================
# 3. Ejecución del scraping
# ==========================================================
driver = iniciar_navegador()
dataset_final = []

try:
    print("Conectando a la fuente de datos...")

    driver.get(URL_OBJETIVO)

    # Esperar a que cargue el contenido dinámico
    time.sleep(5)

    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)

    print(f"Se encontraron {len(elementos)} registros potenciales.")

    for item in elementos:
        try:
            columna_a = item.find_element(
                By.CSS_SELECTOR,
                SELECTOR_DATO_A
            ).text

            columna_b = item.find_element(
              By.CSS_SELECTOR,
              SELECTOR_DATO_B
            ).get_attribute("class")

            # Mantener solo números
            
            valor_limpio = columna_b.replace("star-rating ", "")

            dataset_final.append({
                "variable_precio": columna_a,
                "variable_estrella": valor_limpio,
                "status": 1.0
            })

        except Exception:
            # Saltar elementos vacíos o anuncios
            continue

    # ==========================================================
    # 4. Guardar resultados
    # ==========================================================
    if dataset_final:
        df = pd.DataFrame(dataset_final)

        nombre_archivo = "datos_extraidos_grupo.csv"

        df.to_csv(nombre_archivo, index=False, encoding="utf-8-sig")

        print(f"Proceso exitoso. Archivo generado: {nombre_archivo}")
        print(df.head())

    else:
        print("No se capturaron datos. Revisa los selectores CSS.")

except Exception as e:
    print(f"Error durante el scraping: {e}")

finally:
    driver.quit()
    print("Navegador cerrado.")

Conectando a la fuente de datos...
Se encontraron 20 registros potenciales.
Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
  variable_precio variable_estrella  status
0          £51.77             Three     1.0
1          £53.74               One     1.0
2          £50.10               One     1.0
3          £47.82              Four     1.0
4          £54.23              Five     1.0
Navegador cerrado.
